# Topological observations of a swarm

Turn swarm snapshots into persistence diagrams, fixed-length features and topology through time. This notebook is independent of the swarm-reservoir training pipeline.

## First pass

Run Sections 1–4 for a snapshot, persistence diagram and observation vector. Sections 5–6 compare distances and follow topology through time. The examples use a small swarm and few frames because persistent homology is expensive.

## 1. Generate a swarm trajectory

Generate agent positions and velocities before choosing a distance or filtration.

In [ ]:
include("ABM/load_ABM.jl")
include("ABM/models/Couzin/load_Couzin.jl")
include("TDA/my_TDA.jl")

scenario = Couzin_params_from_preset(:milling; N=50, L=50.0, dt=0.1)
simcfg = SimulationConfig(steps=600, dt=scenario.dt, seed=1)
out = simulate_Couzin_2d(simcfg, scenario.P)
L = scenario.P.L
t_snapshot = length(out.pos_hist)
(frames=length(out.pos_hist), agents=length(out.pos_hist[t_snapshot]), time=out.t[t_snapshot])

## 2. Adapt any swarm state

`swarm_snapshot(source, t)` provides one common TDA-facing state. It always stores positions as an $N\times D$ matrix and, when available, matching velocities and agent labels.

| Source | Call |
|---|---|
| Current reservoir (`res_selected`) | `swarm_snapshot(res_selected)` |
| Couzin or Lund `pos_hist` result | `swarm_snapshot(out, t)` |
| Driven Lymburn, Mizzi or Lund result with `states` | `swarm_snapshot(out_driven, t)` |
| Raw state with `P` and optionally `V` | `swarm_snapshot(raw)` |

The adapter returns physical positions and velocities, not a reservoir feature or readout vector.

In [ ]:
snap = swarm_snapshot(out, t_snapshot)
(agents=size(snap.positions, 1), dimension=size(snap.positions, 2),
    has_velocity=snap.velocities !== nothing, time=snap.time)

In [ ]:
fig_state = Figure(size=(520, 480))
ax_state = Axis(fig_state[1, 1]; xlabel="x", ylabel="y",
    title="Swarm state supplied to TDA", aspect=DataAspect())
scatter!(ax_state, snap.positions[:, 1], snap.positions[:, 2];
    color=:dodgerblue, markersize=8)
xlims!(ax_state, 0, L); ylims!(ax_state, 0, L)
fig_state

## 3. Choose what counts as close

Persistent homology depends on the chosen representation and filtration:

- **position:** physical clustering and spatial holes;
- **position–velocity:** agents are close only when both location and motion are similar;
- **heading dissimilarity:** directional organisation, independent of spatial separation;
- **interaction-path distance:** organisation of the active interaction network.

For a periodic swarm, use minimum-image distance so the coordinate boundary does not split nearby agents.

In [ ]:
D_space = swarm_dissimilarity(out, t_snapshot; periodic=true, L=L)
positive_distances = D_space[D_space .> 0]
εs = collect(range(0, quantile(positive_distances, 0.75); length=70))
(minimum=minimum(positive_distances), maximum=maximum(D_space), ε_max=last(εs))

## 4. From a persistence diagram to an observation vector

As $\varepsilon$ increases, components merge and loops appear and fill. $H_0$ records components; $H_1$ records loops. Distance from the diagonal measures persistence.

In [ ]:
dgm, flt = ph_swarm_snapshot_reps(out; t=t_snapshot, periodic=true, L=L, maxdim=1)

fig_dgm = Figure(size=(850, 380))
for dim in 0:1
    births, deaths, lifetimes = bd_lifetime(dgm[dim + 1])
    upper = isempty(deaths) ? 1.0 : 1.08 * maximum(deaths)
    ax = Axis(fig_dgm[1, dim + 1]; xlabel="birth", ylabel="death",
        title="Persistence diagram: H$(dim)", aspect=DataAspect())
    lines!(ax, [0, upper], [0, upper]; color=:grey55, linestyle=:dash)
    scatter!(ax, births, deaths; color=dim == 0 ? :steelblue : :darkorange, markersize=9)
    xlims!(ax, 0, upper); ylims!(ax, 0, upper)
end
fig_dgm

### Locate a persistent loop

The diagram does not show where a loop occurs. `representative_cycle` extracts its edges and `plot_cycle_on_swarm!` draws them over the agents.

Set `feature_rank=1` for the longest-lived loop; change the rank to inspect another.

In [ ]:
feature_rank = 1   # 1 = longest-lived H1 feature; change and rerun to inspect another

h1 = dgm[2]
births1, deaths1, lifetimes1 = bd_lifetime(h1)
order = sortperm(lifetimes1; rev=true)
feature_idx = order[feature_rank]
chosen = h1[feature_idx]
cycle_edges = representative_cycle(flt, chosen)

fig_cycle = Figure(size=(950, 440))

ax_pd = Axis(fig_cycle[1, 1]; xlabel="birth", ylabel="death",
    title="H1 diagram (rank $feature_rank highlighted)", aspect=DataAspect())
upper1 = 1.08 * maximum(deaths1)
lines!(ax_pd, [0, upper1], [0, upper1]; color=:grey55, linestyle=:dash)
scatter!(ax_pd, births1, deaths1; color=:darkorange, markersize=9)
scatter!(ax_pd, [births1[feature_idx]], [deaths1[feature_idx]];
    color=:crimson, markersize=16, strokewidth=2, strokecolor=:black)
xlims!(ax_pd, 0, upper1); ylims!(ax_pd, 0, upper1)

ax_cyc = Axis(fig_cycle[1, 2]; xlabel="x", ylabel="y",
    title="representative cycle (lifetime=$(round(lifetimes1[feature_idx], digits=2)))",
    aspect=DataAspect())
scatter!(ax_cyc, snap.positions[:, 1], snap.positions[:, 2]; color=:gray70, markersize=6)
plot_cycle_on_swarm!(ax_cyc, snap.positions, cycle_edges; periodic=true, L=L)
xlims!(ax_cyc, 0, L); ylims!(ax_cyc, 0, L)

fig_cycle

In [ ]:
β0 = betti_curve(dgm[1], εs)
β1 = betti_curve(dgm[2], εs)

fig_betti = Figure(size=(760, 360))
ax_betti = Axis(fig_betti[1, 1]; xlabel="ε (minimum-image distance)",
    ylabel="Betti number", title="Topology across spatial scale")
stairs!(ax_betti, εs, β0; label="β₀: components", color=:steelblue, linewidth=2)
stairs!(ax_betti, εs, β1; label="β₁: loops", color=:darkorange, linewidth=2)
axislegend(ax_betti)
fig_betti

In [ ]:
topological_observation = tda_feature_map(out; t=t_snapshot, εs=εs, dims=0:1,
    periodic=true, L=L)

@assert length(topological_observation) == 2 * length(εs)
(feature_length=length(topological_observation),
    first_β0=topological_observation[1],
    first_β1=topological_observation[length(εs) + 1])

`tda_feature_map` concatenates sampled Betti curves into a fixed-length vector:

$$
\mathbf{x}(n)\xrightarrow{\;\phi_{\mathrm{TDA}}\;}\mathbf{z}_{\mathrm{TDA}}(n)\xrightarrow{\;W_{out}\;}\hat{\mathbf y}(n).
$$

Keep the $\varepsilon$ grid, dimensions, representation and boundary treatment fixed between training and testing.

## 5. The same swarm under different observations

Compare physical proximity with heading similarity. Their $\varepsilon$ axes have different units, so compare patterns rather than numerical values.

In [ ]:
D_heading = alignment_dissimilarity(out, t_snapshot)
dgm_heading = ph_snapshot(out; t=t_snapshot, maxdim=1,
    filtration_fn=(o, t) -> alignment_dissimilarity(o, t))
ε_heading = collect(range(0, 2; length=70))

fig_compare = Figure(size=(980, 380))
ax_space = Axis(fig_compare[1, 1]; xlabel="ε_space", ylabel="Betti number",
    title="Physical organisation")
stairs!(ax_space, εs, β0; color=:steelblue, label="β₀")
stairs!(ax_space, εs, β1; color=:darkorange, label="β₁")
axislegend(ax_space)

ax_heading = Axis(fig_compare[1, 2]; xlabel="ε_heading = 1 - cos(Δθ)",
    ylabel="Betti number", title="Directional organisation")
stairs!(ax_heading, ε_heading, betti_curve(dgm_heading[1], ε_heading); color=:steelblue, label="β₀")
stairs!(ax_heading, ε_heading, betti_curve(dgm_heading[2], ε_heading); color=:darkorange, label="β₁")
axislegend(ax_heading)
fig_compare

## 6. Follow topology through time

A CROCKER-style view stacks Betti curves across frames. It shows merging, fragmentation and loops, but does not track individual features between frames.

In [ ]:
t_idxs = unique(round.(Int, range(1, length(out.pos_hist); length=14)))
εs_time = collect(range(0, last(εs); length=45))
crock0 = zeros(Int, length(εs_time), length(t_idxs))
crock1 = similar(crock0)
for (j, t) in enumerate(t_idxs)
    d = ph_swarm_snapshot(out; t=t, periodic=true, L=L, maxdim=1)
    crock0[:, j] = betti_curve(d[1], εs_time)
    crock1[:, j] = betti_curve(d[2], εs_time)
end
times = [swarm_time(out, t) for t in t_idxs]

fig_time = Figure(size=(950, 620))
ax0 = Axis(fig_time[1, 1]; xlabel="time", ylabel="ε", title="β₀(ε,t): components")
hm0 = heatmap!(ax0, times, εs_time, permutedims(crock0); colormap=:viridis)
Colorbar(fig_time[1, 2], hm0)
ax1 = Axis(fig_time[2, 1]; xlabel="time", ylabel="ε", title="β₁(ε,t): loops")
hm1 = heatmap!(ax1, times, εs_time, permutedims(crock1); colormap=:magma)
Colorbar(fig_time[2, 2], hm1)
fig_time

## 7. Topological observation layer

The fixed-length features could feed the swarm-reservoir readout, but this connection has not yet been implemented or tested.

## Checks before using topological features

1. **Geometry:** match periodic or unbounded distance to the swarm model.
2. **Scale:** choose an $\varepsilon$ grid that covers informative births and deaths without spending most samples where nothing changes.
3. **Feature consistency:** use the identical representation, grid and dimensions for training and testing.
4. **Temporal sampling:** match frame spacing to the swarm response time.
5. **Cost:** begin with $H_0/H_1$, modest $N$ and sparse time sampling.
6. **Interpretation:** topology describes the chosen relation, not necessarily the swarm mechanism or task.

## Next step

Compare these summaries with the existing observations before adding them to `Tutorial_swarmRC.ipynb`.